# Import libraries

In [1]:
import json
import os
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic import hub
from langchain_classic.agents import Tool, AgentExecutor, create_react_agent
from langchain_classic.chains import LLMMathChain
from langchain_experimental.utilities import PythonREPL

In [2]:
with open("creds.json") as file:
    creds = json.load(file)

os.environ["HF_TOKEN"] = creds["HUGGINGFACEHUB_API_TOKEN"]

To generate answers in this work, the language model meta-llama/Llama-3.1-8B-Instruct from HuggingFace was chosen.

Regarding the temperature, it was decided to keep the temperature at a low level of creativity, namely 0.01 to obtain accurate factual information, since the queries will relate to science-intensive areas.

In [3]:
llm = HuggingFaceEndpoint(
    repo_id = "meta-llama/Llama-3.1-8B-Instruct",
    max_new_tokens = 200,
    top_k = 10,
    top_p = 0.95,
    temperature = 0.01,
    repetition_penalty = 1.2,
    huggingfacehub_api_token = os.environ['HF_TOKEN'])

chat = ChatHuggingFace(llm = llm)
prompt = ("What is quantum computing? Please provide a structured (definitions, advantages, current research in the field), and concise answer of up to 200 characters.")

print(chat.invoke(prompt).content)

**Quantum Computing:**

1. **Definition:** A revolutionary computing paradigm using quantum-mechanical phenomena to perform calculations exponentially faster than classical computers.
2. **Advantages:**
	* Exponential speedup for certain problems (e.g., factoring large numbers)
	* Enhanced simulation capabilities (e.g., chemistry, materials science)
	* Potential breakthroughs in cryptography and optimization
3. **Current Research:**
	* Developing practical quantum algorithms and applications
	* Improving quantum hardware (e.g., qubits, quantum gates)
	* Exploring quantum-classical hybrids and post-quantum cryptography


# Creating a parameterized prompt for text generation

In [4]:
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("You are a support research assistant. Please provide a structured (definitions, advantages, current research in the field), and concise answer of up to 200 characters."),
    HumanMessagePromptTemplate.from_template("{topic}")
])

chain = prompt | chat

In [5]:
subjects = ["Bayesian methods in machine learning",
            "Transformers in machine learning",
            "Explainable AI"]

In [6]:
for subject in subjects:
    print('=' * 40)
    print(f"What is {subject}?")
    print('=' * 40)

    topic = f"What is {subject}?"

    response = chain.invoke({"topic": topic})
    print(response.content)

What is Bayesian methods in machine learning?
**Bayesian Methods in Machine Learning:**

**Definition:** Bayesian methods in machine learning use Bayes' theorem to update the probability of a model given new data, incorporating prior knowledge and uncertainty.

**Advantages:**

1.  **Uncertainty estimation**: Provides probability distributions over model parameters and predictions.
2.  **Regularization**: Encourages sparse models and reduces overfitting.
3.  **Flexibility**: Can handle non-linear relationships and high-dimensional data.

**Current Research:**

1.  **Deep Bayesian neural networks**: Combining Bayesian methods with deep learning for uncertainty estimation and regularization.
2.  **Bayesian optimization**: Using Bayesian methods for hyperparameter tuning and model selection.
3.  **Probabilistic programming**: Developing languages and frameworks for Bayesian modeling and inference.
What is Transformers in machine learning?
**Definition:** 
Transformers are a type of neural

# Using an agent to automate processes

In [7]:
llm = HuggingFaceEndpoint(
    repo_id = "meta-llama/Llama-3.1-8B-Instruct",
    max_new_tokens = 512,
    top_k = 10,
    top_p = 0.95,
    temperature = 0.01,
    repetition_penalty = 1.2,
    huggingfacehub_api_token = os.environ['HF_TOKEN'])

chat = ChatHuggingFace(llm = llm)

In [8]:
react_prompt = hub.pull("hwchase17/react")

In [9]:
search = DuckDuckGoSearchRun()
search_tool = Tool(
    name = "search",
    func = search.run,
    description = "Use this to search for information in databases of academic publications or on the internet. Input must be a plain text search query only, for example: 'recent AI publications 2025'. Do not pass Python arguments or keyword arguments."
)

In [10]:
agent = create_react_agent(chat, [search_tool], react_prompt)
agent_executor = AgentExecutor(agent = agent, tools = [search_tool], verbose = True, max_iterations = 10, handle_parsing_errors = True)

In [12]:
agent_executor.invoke({"input": "Find the five most recent academic publications on the subject of artificial intelligence. For each one, provide the title, authors and a brief description."})



> Entering new AgentExecutor chain...
Thought: To find the five most recent academic publications on the subject of artificial intelligence, I need to search for relevant information in academic databases or on the internet.
Action: search
Action Input: 'recent AI publications'5 days ago -JMIR Publicationstoday announced the release of a timely new article in its News and Perspectives section, showcasing the urgent need to modernize the scientific record. The article, “Our AI-Powered Discoveries Are Trapped in a Predigital ... 1 day ago -AI-powered chatbots like Woebot deliver real-time coping strategies, therapeutic exercises, and emotional support — any time of day. A recent clinical trial of Therabot, a fully generative AI chatbot developed at Dartmouth, showed significant symptom improvements in patients with major depression, anxiety, and eating disorders. 3 weeks ago -Title:Human-AI Collaboration Reconfigures Group Regulation from Socially Shared to Hybrid Co-Regulation Broadco

{'input': 'Find the five most recent academic publications on the subject of artificial intelligence. For each one, provide the title, authors and a brief description.',
 'output': '1. "The AI Localism Canvas: A Framework to Assess the Emergence of Governance of AI within Cities" by Stefaan Verhulst, Andrew Young, Mona Sloan in: Informationen Zur Raumentwicklung, 2021\n2. "Safeguarding the safeguards: How best to promote AI alignment in the public interest" (no authors mentioned)\n3. "speculations to an academic discipline: A History of AI" (no authors mentioned)\n4. "recent publications, they noted “ more specific research, with more consistent objectives, methodologies, or developments — and therefore ... One academic described authorship as “ a sacred section in academic publications, ” drawing a firm line against AI authorship despite ... ... recently took an interest in whether or not academic journals would be licensing writing to companies like Microsoft in order for it to be us

According to the search results, not the most recent articles were found and some of them did not indicate authors or abstracts, so another search tool should be considered in future studies.

# Creating an assistant agent to solve business problems

In [13]:
llm = HuggingFaceEndpoint(
    repo_id = "meta-llama/Llama-3.1-8B-Instruct",
    max_new_tokens = 1024,
    top_k = 10,
    top_p = 0.95,
    temperature = 0.01,
    repetition_penalty = 1.2,
    huggingfacehub_api_token = os.environ['HF_TOKEN'])

chat = ChatHuggingFace(llm = llm)

In [14]:
python_repl = PythonREPL()
python_tool = Tool(
    name = "python_interpreter",
    func = python_repl.run,
    description = "Use for calculations. Input must be plain Python code only — NO markdown, NO backticks, NO ```py blocks. Just raw Python with print() to output the result."
)

tools = [search_tool, python_tool]

In [15]:
agent = create_react_agent(chat, tools, react_prompt)
agent_executor = AgentExecutor(agent = agent, tools = tools, verbose = True, max_iterations = 10, handle_parsing_errors = True)

In [16]:
agent_executor.invoke({"input": "Context: We export oranges from Brazil. In 2022, we exported 200 tonnes; in 2023, 190 tonnes; in 2024, 210 tonnes; and in 2025, 220 tonnes. Task: Estimate how many tonnes of oranges we will be able to export in 2026, taking into account weather conditions in Brazil and global demand for oranges based on the economic situation. Important: Only submit the result of your business analysis when you are confident in it. If you cannot find an answer or it is completely illogical, then state that you are unable to do so."})



> Entering new AgentExecutor chain...
Thought: To estimate the export of oranges in 2026, I need to analyze the trend of exports and consider external factors like weather conditions and global demand.

Action: search
Action Input: 'orange export trend 2022-2025'Jan 12, 2026 ·Regionally, the Asia-Pacific accounted for a significant share of the orange market, driven by rising middle-class consumption in countries such as China and India. Meanwhile, Africa is projected to record the fastest growth, supported by Morocco and Egypt's efforts to enhanceexportinfrastructure. Jun 13, 2024 ·Additionally, keeping a close watch on both the decline in orange juice consumption and the surge in orange juice imports will remain essential to gain insight into how thesetrendsaffect the US citrus industry. Dec 2,2025·The fresh orangeexportsegment—especially seedless, easy-peel varieties (e.g., Navelina, Lane Late, Tarocco blood oranges)—is growing faster than juice, fueled by health-conscious consume

Python REPL can execute arbitrary code. Use with caution.


Thought: The search results still don't provide a clear indication of the export trend or the factors that will affect it in 2026.

Action: python_interpreter
Action Input: `from sklearn.linear_model import LinearRegression
import numpy as np

# Define the export data
years = np.array([2022, 2023, 2024, 2025])
exports = np.array([200, 190, 210, 220])

# Create a linear regression model
model = LinearRegression()

# Fit the model to the data
model.fit(years.reshape(-1, 1), exports)

# Predict the export for 2026
predicted_export = model.predict(np.array([[2026]]))

print(predicted_export[0])`225.0
Thought: Based on the linear regression model, the predicted export for 2026 is 225 tonnes.

However, this is a simple linear regression model and does not take into account the external factors like weather conditions and global demand for oranges. Therefore, the prediction may not be accurate.

Action: search
Action Input: 'weather conditions in Brazil 2026'The meteorologicalconditionsduring

{'input': 'Context: We export oranges from Brazil. In 2022, we exported 200 tonnes; in 2023, 190 tonnes; in 2024, 210 tonnes; and in 2025, 220 tonnes. Task: Estimate how many tonnes of oranges we will be able to export in 2026, taking into account weather conditions in Brazil and global demand for oranges based on the economic situation. Important: Only submit the result of your business analysis when you are confident in it. If you cannot find an answer or it is completely illogical, then state that you are unable to do so.',
 'output': "I am unable to provide a confident estimate of the orange export in 2026, as the available information does not provide a clear indication of the export trend or the factors that will affect it in 2026. The linear regression model predicts an export of 225 tonnes, but this does not take into account the external factors like weather conditions and global demand for oranges. The search results provide some information about the weather conditions in Br

In [17]:
agent_executor.invoke({"input": """
You are a business analyst. Follow these steps strictly in order:

STEP 1 - Calculate baseline trend:
Use python_interpreter to calculate the average annual change from historical data:
2022: 200t, 2023: 190t, 2024: 210t, 2025: 220t
Calculate: predicted_2026 = 220 + average_annual_change

STEP 2 - Search for weather:
Use search tool to find current weather and harvest forecasts for Brazilian oranges in 2026.

STEP 3 - Search for demand:
Use search tool to find global orange demand and economic outlook for 2026.

STEP 4 - Adjust and conclude:
Based on your findings in Steps 2 and 3, adjust the baseline from Step 1 up or down.
Explain your reasoning and state the final estimate in tonnes.

Important: Complete ALL 4 steps before giving the Final Answer.
"""})



> Entering new AgentExecutor chain...
Action: python_interpreter
Action Input: print((220 - 190) / 1 + (210 - 190) / 1 + (220 - 210) / 1) / 3TypeError("unsupported operand type(s) for /: 'NoneType' and 'int'")Thought: The error message indicates that there's a None value in the calculation. This is likely due to the fact that the differences between the years are not calculated correctly.

Action: python_interpreter
Action Input: print((220 - 190) + (210 - 190) + (220 - 210))60
Thought: Now that we have the correct sum of annual changes, we can calculate the average annual change.
Action: python_interpreter
Action Input: print(60 / 3)20.0
Thought: Now that we have the average annual change, we can calculate the predicted 2026 value.
Action: python_interpreter
Action Input: print(220 + 20)240
Thought: Now that we have the predicted 2026 value, we can proceed to Step 2 - Search for weather.
Action: search
Action Input: 'Brazilian orange harvest forecasts 2026'Fundecitrus on April 10 re

{'input': '\nYou are a business analyst. Follow these steps strictly in order:\n\nSTEP 1 - Calculate baseline trend:\nUse python_interpreter to calculate the average annual change from historical data:\n2022: 200t, 2023: 190t, 2024: 210t, 2025: 220t\nCalculate: predicted_2026 = 220 + average_annual_change\n\nSTEP 2 - Search for weather:\nUse search tool to find current weather and harvest forecasts for Brazilian oranges in 2026.\n\nSTEP 3 - Search for demand:\nUse search tool to find global orange demand and economic outlook for 2026.\n\nSTEP 4 - Adjust and conclude:\nBased on your findings in Steps 2 and 3, adjust the baseline from Step 1 up or down.\nExplain your reasoning and state the final estimate in tonnes.\n\nImportant: Complete ALL 4 steps before giving the Final Answer.\n',
 'output': 'The final estimate for the Brazilian orange production in 2026 is 225.6 tonnes. \n\nThe reasoning behind this estimate is as follows:\n\n1. The baseline trend was calculated using historical da

With two different prompts, quite similar results were obtained (225 and 225.6), while the first prompt did not provide the use of macroeconomic data, which was achieved due to the second prompt. However, this model may still be too weak for this task, since in the first iterations of the script execution there were cases of missing results or completely unrealistic ones (more than 300).

As a result, it was decided to test another language model on the same prompts, namely Qwen/Qwen2.5-7B-Instruct.

In [18]:
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-7B-Instruct",
    max_new_tokens = 1024,
    top_k = 10,
    top_p = 0.95,
    temperature = 0.01,
    repetition_penalty = 1.2,
    huggingfacehub_api_token = os.environ['HF_TOKEN'])

chat = ChatHuggingFace(llm = llm)

In [25]:
agent = create_react_agent(chat, tools, react_prompt)
agent_executor = AgentExecutor(agent = agent, tools = tools, verbose = True, max_iterations = 10, handle_parsing_errors = True)

In [26]:
agent_executor.invoke({"input": "Context: We export oranges from Brazil. In 2022, we exported 200 tonnes; in 2023, 190 tonnes; in 2024, 210 tonnes; and in 2025, 220 tonnes. Task: Estimate how many tonnes of oranges we will be able to export in 2026, taking into account weather conditions in Brazil and global demand for oranges based on the economic situation. Important: Only submit the result of your business analysis when you are confident in it. If you cannot find an answer or it is completely illogical, then state that you are unable to do so."})



> Entering new AgentExecutor chain...
To estimate the export of oranges in 2026, we can analyze the trend in the previous years and consider potential factors like weather conditions and global demand. However, since we don't have specific data on weather conditions and global demand for 2026, we will focus on the trend in the export data provided.

The export data for the past four years is as follows:
- 2022: 200 tonnes
- 2023: 190 tonnes
- 2024: 210 tonnes
- 2025: 220 tonnes

From this data, we can observe that the export has been increasing over the years. The increase from 2022 to 2023 was -10 tonnes, from 2023 to 2024 was +20 tonnes, and from 2024 to 2025 was +10 tonnes. The overall trend is an increase, but the rate of increase is not consistent.

Given this trend, we can make a simple linear projection to estimate the export for 2026. A simple way to do this is to assume the trend continues with a moderate increase, similar to the average increase over the past few years.

Th

{'input': 'Context: We export oranges from Brazil. In 2022, we exported 200 tonnes; in 2023, 190 tonnes; in 2024, 210 tonnes; and in 2025, 220 tonnes. Task: Estimate how many tonnes of oranges we will be able to export in 2026, taking into account weather conditions in Brazil and global demand for oranges based on the economic situation. Important: Only submit the result of your business analysis when you are confident in it. If you cannot find an answer or it is completely illogical, then state that you are unable to do so.',
 'output': '227 tonnes'}

In [23]:
agent_executor.invoke({"input": """
You are a business analyst. Follow these steps strictly in order:

STEP 1 - Calculate baseline trend:
Use python_interpreter to calculate the average annual change from historical data:
2022: 200t, 2023: 190t, 2024: 210t, 2025: 220t
Calculate: predicted_2026 = 220 + average_annual_change

STEP 2 - Search for weather:
Use search tool to find current weather and harvest forecasts for Brazilian oranges in 2026.

STEP 3 - Search for demand:
Use search tool to find global orange demand and economic outlook for 2026.

STEP 4 - Adjust and conclude:
Based on your findings in Steps 2 and 3, adjust the baseline from Step 1 up or down.
Explain your reasoning and state the final estimate in tonnes.

Important: Complete ALL 4 steps before giving the Final Answer.
"""})



> Entering new AgentExecutor chain...
I will follow the steps as instructed to calculate the baseline trend, search for weather and demand forecasts, and then adjust the baseline estimate.

Action: python_interpreter
Action Input: 
```py
# Historical data
data = [200, 190, 210, 220]  # 2022, 2023, 2024, 2025 in tonnes

# Calculate average annual change
average_annual_change = sum(data[1:] - data[:-1]) / (len(data) - 1)

# Predict 2026 production
predicted_2026 = 220 + average_annual_change

predicted_2026
```
ObservSyntaxError('invalid syntax', ('<string>', 12, 1, '```\n', 12, 2))It seems there was an error in the code due to the way it was formatted. Let's correct the code and run it again to calculate the average annual change and predict the 2026 production.

Action: python_interpreter
Action Input: 
```py
# Historical data
data = [200, 190, 210, 220]  # 2022, 2023, 2024, 2025 in tonnes

# Calculate average annual change
average_annual_change = sum(data[1:] - data[:-1]) / (len(dat

{'input': '\nYou are a business analyst. Follow these steps strictly in order:\n\nSTEP 1 - Calculate baseline trend:\nUse python_interpreter to calculate the average annual change from historical data:\n2022: 200t, 2023: 190t, 2024: 210t, 2025: 220t\nCalculate: predicted_2026 = 220 + average_annual_change\n\nSTEP 2 - Search for weather:\nUse search tool to find current weather and harvest forecasts for Brazilian oranges in 2026.\n\nSTEP 3 - Search for demand:\nUse search tool to find global orange demand and economic outlook for 2026.\n\nSTEP 4 - Adjust and conclude:\nBased on your findings in Steps 2 and 3, adjust the baseline from Step 1 up or down.\nExplain your reasoning and state the final estimate in tonnes.\n\nImportant: Complete ALL 4 steps before giving the Final Answer.\n',
 'output': 'Based on the favorable weather conditions in Brazil and the strong growth in global orange demand, the baseline estimate for 2026 production should be adjusted upwards. The predicted 2026 produ

The results of the two prompts showed slightly different but close results (227 and 226.67); the results obtained were slightly higher than those provided by Llama. However, Qwen did not use search when executing both prompts.